# Modelling to Generate Alternatives (MGA) for a multi-regional energy system

The cost-optimal design of an energy system is rarely the only sensible one. Often, many designs whose costs are only slightly higher differ considerably in their technology mix. Modelling to Generate Alternatives (MGA) explores this near-optimal solution space. In ETHOS.FINE, MGA is available with the function `fn.runMGA`, which works in four steps:
1. The cost-optimal solution is computed with `esM.optimize`.
2. A constraint limits the total annual cost to the optimal cost plus a slack (e.g. 10 %).
3. The cost objective is replaced by a weighted sum of capacity and/or operation variables, which is either minimized or maximized.
4. The already declared optimization problem is solved again for every alternative.

The multi-regional energy system of example 03 is used here. The workflow is structured as follows:
1. Required packages are imported and the input data path is set
2. The energy system model of example 03 is built
3. The time series are aggregated
4. The cost-optimal solution is computed
5. Alternatives are generated with MGA
6. The alternatives are compared

# 1. Import required packages and set input data path

The ETHOS.FINE framework is imported which provides the required classes and functions for modeling the energy system.

In [1]:
import fine as fn
from getData import getData
import pandas as pd

data = getData()

%matplotlib inline
%load_ext autoreload
%autoreload 2

# 2. Build the energy system model

The energy system model is identical to the one built step by step in example 03 (commodity sources, conversion, storage, transmission and sinks). To keep this notebook focused on MGA, that setup is factored out into `getModel.py`; see example 03 for a detailed walkthrough of each component.

In [2]:
from getModel import getModel

esM = getModel(data)

# 3. Temporal aggregation

In [3]:
esM.aggregateTemporally(n_clusters=7)

# 4. Cost-optimal optimization

MGA starts from the cost-optimal solution, so the model is optimized first. The keyword arguments of `esM.optimize` are stored in `solverKwargs` because `fn.runMGA` needs the same ones for every MGA solve. In particular, `timeSeriesAggregation=True` has to be passed there again, otherwise the results of the alternatives are post-processed as if no time series aggregation was used.

In [4]:
# The `optimizationSpecs` only work with the Gurobi solver. If you are using another solver you need to choose
# specs specific to this solver or no specs.
solverKwargs = dict(
    timeSeriesAggregation=True,
    optimizationSpecs="OptimalityTol=1e-3 method=2 cuts=0 MIPGap=5e-3",
    solver=fn.utils.ImplementedSolvers.GUROBI.value,
)
esM.optimize(**solverKwargs)

Set parameter Threads to value 3


Set parameter LogFile to value ""


Set parameter OptimalityTol to value 0.001


Set parameter Method to value 2


Set parameter Cuts to value 0


Set parameter MIPGap to value 0.005


... (103 outputs left out)

Solution count 1: 53.2964 


Optimal solution found (tolerance 5.00e-03)


Best objective 5.329644059174e+01, best bound 5.318564797346e+01, gap 0.2079%


C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Li-ion batteries
  warnings.warn(
C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Pumped hydro storage
  warnings.warn(
C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (biogas)
  warnings.warn(
C:\Users\l.wijesinghe\Hackathon\FINE\fine\storage.py:2230: UserWarning: Charge and discharge at the same time for component Salt caverns (hydrogen)
  warnings.warn(


# 5. Generate alternatives with MGA

The MGA run is specified by a dictionary:
- `slack`: allowed increase of the total annual cost compared to the cost-optimal solution (here 10 %).
- `objectives`: one entry per alternative. Each objective has a `name` (also used for the Excel file name) and a `sense` (`"min"` or `"max"`). The `variableSelection` defines which variables enter the objective:
    - `"custom"`: only the variables listed in `terms`. A term consists of a `component`, a `variable` (`"capacity"`, `"operation"`, `"charge"` or `"discharge"`) and a `weight`. Optionally, it can be restricted to some `locations` and `investmentPeriods`.
    - `"allCapacity"`: all capacity variables of the model.
    - `"allCapacityAndOperation"`: all capacity and operation variables of the model.

    For `"allCapacity"` and `"allCapacityAndOperation"`, the weights can be adapted with `defaultWeight`, `weights` and `excludeComponents`. Operation variables are summed up to annual values.

- `persistent`: if `True`, the model is transferred to a persistent Gurobi solver once and every alternative only replaces the objective function, instead of passing the complete model to the solver again for every alternative. This speeds up runs with several alternatives, but requires `solver="gurobi"`. Default: `False`.

Here, two alternatives are generated. Both minimize a weighted sum of the onshore wind capacity, the PV generation, the pumped hydro storage discharge and the transmission capacities. The second alternative puts a higher weight on the storage discharge and the transmission capacities. Note that the operation variables are annual sums (in GWh) and are therefore much larger than the capacities (in GW). With the weights chosen here, the PV generation dominates both objectives. The weights should be chosen with these magnitudes in mind.

With `save_mga_results_in_excel=True`, the cost-optimal solution and every alternative are written to Excel files in the folder `OutputData`. All further keyword arguments (here `solverKwargs`) are passed to `esM.optimize`.

In [5]:
mgaSpec = {
    "slack": 0.1,
    "objectives": [
        {
            "name": "alternative1",
            "sense": "min",
            "variableSelection": "custom",
            "terms": [
                {"component": "Wind (onshore)", "variable": "capacity", "weight": 0.3},
                {"component": "PV", "variable": "operation", "weight": 0.6},
                {
                    "component": "Pumped hydro storage",
                    "variable": "discharge",
                    "weight": 0.01,
                },
                {"component": "AC cables", "variable": "capacity", "weight": 0.01},
                {"component": "DC cables", "variable": "capacity", "weight": 0.01},
                {
                    "component": "Pipelines (biogas)",
                    "variable": "capacity",
                    "weight": 0.01,
                },
                {
                    "component": "Pipelines (hydrogen)",
                    "variable": "capacity",
                    "weight": 0.01,
                },
            ],
        },
        {
            "name": "alternative2",
            "sense": "min",
            "variableSelection": "custom",
            "terms": [
                {"component": "Wind (onshore)", "variable": "capacity", "weight": 0.3},
                {"component": "PV", "variable": "operation", "weight": 0.6},
                {
                    "component": "Pumped hydro storage",
                    "variable": "discharge",
                    "weight": 0.9,
                },
                {"component": "AC cables", "variable": "capacity", "weight": 0.8},
                {"component": "DC cables", "variable": "capacity", "weight": 0.4},
                {
                    "component": "Pipelines (biogas)",
                    "variable": "capacity",
                    "weight": 0.67,
                },
                {
                    "component": "Pipelines (hydrogen)",
                    "variable": "capacity",
                    "weight": 0.12,
                },
            ],
        },
    ],
}

results = fn.runMGA(
    esM,
    spec=mgaSpec,
    persistent=True,
    save_mga_results_in_excel=True,
    excelOutputDir="OutputData",
    excelFilePrefix="MGA",
    **solverKwargs,
)

MGA run: slack 10.0%, 2 alternative(s), persistent Gurobi


Excel output enabled: C:\Users\l.wijesinghe\Hackathon\FINE\examples\12_Modelling_to_Generate_Alternatives\OutputData


Transferring the model to Gurobi (once) ...


Model transferred (3.2 s).


Accessing the cost-optimal solution ...


Cost-optimal solution: total cost = 53.2964


... (377 outputs left out)

  Writing results to Excel: OutputData\MGA_alternative2.xlsx ...


  Excel file written (28.3 s).


Alternative 'alternative2' done (224.5 s).


MGA run finished in 378.4 s.


# 6. Compare the alternatives

`fn.runMGA` returns a dictionary with one entry for the cost-optimal solution (`"costOptimal"`) and one entry per alternative. Each entry contains the total annual cost (`"cost"`), the optimization summaries of all modeling classes (`"summary"`) and, for the alternatives, the value of the MGA objective (`"mgaObjective"`).

All alternatives have a total annual cost of at most 110 % of the cost-optimal solution:

In [6]:
pd.DataFrame(
    {
        name: {
            "TAC [1e9 Euro/a]": result["cost"],
            "MGA objective": result.get("mgaObjective"),
        }
        for name, result in results.items()
    }
)

,costOptimal,alternative1,alternative2
TAC [1e9 Euro/a],53.296441,58.626085,58.626085
MGA objective,NaN,40348.235970,43928.552777


The installed capacities, summed over all locations, show how differently the alternatives are designed:

In [7]:
def totalCapacities(summaries):
    """Return the capacity of each component, summed over all locations."""
    capacities = []
    for modelingClass, summary in summaries.items():
        capacity = (
            summary.xs("capacity", level="Property")
            .groupby(level=["Component", "Unit"])
            .sum()
            .sum(axis=1)
        )
        if modelingClass == "TransmissionModel":
            capacity /= 2  # the summary contains both directions of each connection
        capacities.append(capacity)
    return pd.concat(capacities)


capacities = pd.DataFrame(
    {name: totalCapacities(result["summary"]) for name, result in results.items()}
)
capacities.round(2)

,,costOptimal,alternative1,alternative2
Component,Unit,,,
Existing run-of-river plants,[GW$_{el}$],3.811,3.811,3.811
PV,[GW$_{el}$],189.671773,94.529549,102.310839
Wind (offshore),[GW$_{el}$],81.0,81.0,81.0
Wind (onshore),[GW$_{el}$],36.01065,128.682629,127.430012
Electrolyzer,[GW$_{el}$],29.318816,31.619272,28.977027
New CCGT plants (biogas),[GW$_{el}$],10.830446,12.238233,12.896641
Li-ion batteries,[GW$_{el}$*h],128.399536,75.458894,94.501296
Pumped hydro storage,[GW$_{el}$*h],76.937,76.937,76.937
Salt caverns (biogas),"[GW$_{biogas,LHV}$*h]",6421.665743,14337.775039,10900.311254
